In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import recall_score, f1_score, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


Device: cpu


In [2]:
# load dataset
import pandas as pd
import os

file_path = os.path.join("..", "data", "AIML Dataset.csv")

df = pd.read_csv(file_path, header=None)

df.columns = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud",
    "isFlaggedFraud"
]

df["step"] = df["step"].astype(str).str.extract(r'(\d+)').astype(int)
# Inspect data
print(df.head())
print(df.dtypes)


   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  
step                int64
type               object
amount            float64
nameOrig           object
o

In [3]:
df = df.sort_values(by="step").reset_index(drop=True)

In [4]:
le = LabelEncoder()
df["type"] = le.fit_transform(df["type"])

In [5]:
features = ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
x = df[features].values
y = df["isFraud"].values

print(f"Data Loaded: {len(df)} records")

Data Loaded: 6362620 records


In [6]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

In [7]:
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
x_train_t = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
x_test_t = torch.tensor(x_test, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [9]:
train_loader = DataLoader(TensorDataset(x_train_t, y_train_t), batch_size=2048, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test_t, y_test_t), batch_size=2048)

In [10]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.999, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

criterion = FocalLoss()


In [11]:
class FraudLSTM(nn.Module):
    def __init__(self, input_size):
        super(FraudLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=64,
            num_layers=2,
            batch_first=True
        )
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]
        out = self.fc(out)
        return out

model = FraudLSTM(len(features)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)


FraudLSTM(
  (lstm): LSTM(7, 64, num_layers=2, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [12]:
epochs = 5
model.train()

for epoch in range(epochs):
    epoch_loss = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.6f}")


Epoch 1 | Loss: 0.004356
Epoch 2 | Loss: 0.000955
Epoch 3 | Loss: 0.000769
Epoch 4 | Loss: 0.000710
Epoch 5 | Loss: 0.000679


In [13]:
model.eval()
preds = []

with torch.no_grad():
    for batch_X, _ in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        probs = torch.sigmoid(outputs)
        preds.extend(probs.cpu().numpy())

y_pred = (np.array(preds) > 0.5).astype(int)

print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Recall: 0.7133
Confusion Matrix:
[[1270692     189]
 [    471    1172]]
